# 🧰 L5 — Écrire et consommer un serveur MCP local

> ⏱️ **Durée indicative : 30 à 45 minutes**  
> 🎓 **Niveau : débutant** — nous allons lire, comprendre puis consommer un vrai serveur MCP local.

Dans la leçon précédente, nous avons donné une fonction Python à un agent. Ici, nous allons consommer un **serveur MCP local** (`mcp_chinook.py`) qui expose la base Chinook via deux outils, puis le brancher sur un agent LangChain.

📚 Dès maintenant, gardez sous la main la [documentation MCP officielle de LangChain](https://docs.langchain.com/oss/python/langchain/mcp).

## 🎯 Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- lire le code source d'un serveur MCP et identifier ses outils ;
- distinguer le **serveur**, l'**adaptateur** et le **transport** ;
- découvrir les outils exposés par un serveur MCP et lire leur schéma ;
- connecter ces outils à un agent créé avec `create_agent` ;
- repérer précisément ce que décide Mistral et ce qu'exécute réellement Python.

### 🧭 Notre signalétique

🎯 objectif · 🧠 intuition · 🗺️ schéma mental · 🛠️ construction · 🔮 prédiction · ▶️ exécution · 👀 observation · ⚠️ piège · 🧪 exercice · ✅ correction · 📚 documentation

## 🗺️ La carte du voyage

```text
Question de l'utilisateur
          │
          ▼
🤖 Agent LangChain ──► 🧠 Mistral choisit un outil et ses arguments
          │
          ▼
🔌 Adaptateur LangChain MCP
          │  transport stdio (messages via entrée/sortie standard)
          ▼
🧰 mcp_chinook.py ──► 🗄️ lit Chinook.db en lecture seule
          │
          ▼
Résultat renvoyé à Mistral, puis réponse finale
```

🧠 **Analogie ELI5 adulte :** l'agent est un analyste, l'adaptateur est le standardiste, `mcp_chinook.py` est la base de données avec son interface, et le transport `stdio` est le câble. L'analyste pose la question ; le serveur interroge SQLite et répond.

## 🔐 1. Préparer Mistral sans exposer de secret

Ce notebook charge les variables d'environnement depuis `.env` et utilise :

- `MISTRAL_API_KEY` — lu automatiquement par `ChatMistralAI` ;
- `MISTRAL_SERVER_URL` — passé à `endpoint`.

Aucune valeur n'est affichée.

📚 `ChatMistralAI` et ses paramètres sont décrits dans l'[intégration officielle LangChain–Mistral](https://docs.langchain.com/oss/python/integrations/chat/mistralai).

In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())
import os

from langchain_mistralai import ChatMistralAI

# 📚 https://docs.langchain.com/oss/python/integrations/chat/mistralai
mistral_model = ChatMistralAI(
    model="mistral-medium-latest",
    temperature=0,
    endpoint=os.environ["MISTRAL_SERVER_URL"] + "/v1",
)
print("✅ Configuration Mistral détectée — aucune valeur n'est affichée.")

✅ Configuration Mistral détectée — aucune valeur n'est affichée.


## 🧰 2. Point de comparaison : un outil Python local

Un **outil local** est une fonction qui vit dans le même programme que notre agent. Le décorateur [`@tool`](https://docs.langchain.com/oss/python/langchain/tools) transforme sa signature et sa docstring en une fiche lisible par le modèle : nom, description et schéma des arguments.

Ici, Python connaît déjà les fuseaux horaires grâce à sa bibliothèque standard. Aucun serveur externe n'est nécessaire.

### 🔮 Pause prédiction

Qui exécutera réellement `local_current_time` : Mistral ou Python ?

➡️ Formulez votre réponse avant d'exécuter les deux cellules suivantes.

In [2]:
from datetime import datetime
from zoneinfo import ZoneInfo

from langchain.tools import tool


# 📚 @tool : https://docs.langchain.com/oss/python/langchain/tools
@tool
def local_current_time(timezone: str) -> str:
    """Return the current time for an IANA timezone such as Europe/Paris."""
    now = datetime.now(ZoneInfo(timezone))
    return now.isoformat(timespec="seconds")

In [3]:
import json

print("Nom :", local_current_time.name)
print("Description :", local_current_time.description)
print("Schéma :")
print(json.dumps(local_current_time.args_schema.model_json_schema(), indent=2, ensure_ascii=False))
print("Résultat direct :", local_current_time.invoke({"timezone": "Europe/Paris"}))

Nom : local_current_time
Description : Return the current time for an IANA timezone such as Europe/Paris.
Schéma :
{
  "description": "Return the current time for an IANA timezone such as Europe/Paris.",
  "properties": {
    "timezone": {
      "title": "Timezone",
      "type": "string"
    }
  },
  "required": [
    "timezone"
  ],
  "title": "local_current_time",
  "type": "object"
}


Résultat direct : 2026-09-02T14:01:05+02:00


### 👀 Ce qu'il faut observer

- LangChain a fabriqué le **contrat** de l'outil à partir du code Python.
- Le paramètre `timezone` apparaît dans un schéma JSON.
- Lors de l'appel direct, c'est bien **Python** qui exécute la fonction.
- Mistral n'a encore rien décidé : nous n'avons pas appelé le modèle.

🧠 Un outil MCP offrira le même type de contrat à LangChain, mais son code vivra dans un autre processus.

## 🔌 3. MCP en trois pièces faciles à distinguer

### 🧰 Le serveur MCP

Le serveur est le programme qui **possède et exécute** les capacités. Ici, c'est `mcp_chinook.py` : un fichier Python lisible que vous pouvez modifier. Il expose deux outils : `list_tables` et `query_chinook`.

### 🔌 L'adaptateur LangChain

[`MultiServerMCPClient`](https://docs.langchain.com/oss/python/langchain/mcp) se connecte au serveur, découvre ses outils et les convertit en outils compris par LangChain. Malgré son nom, il fonctionne aussi avec un seul serveur.

### 🚇 Le transport `stdio`

Le transport est la route empruntée par les messages. Avec `stdio`, LangChain lance `mcp_chinook.py` comme un sous-processus et échange des messages structurés via son entrée et sa sortie standard.

> 🧠 **À retenir :** MCP standardise la conversation entre programmes. Ce même serveur pourrait être exposé en HTTP avec `mcp.run(transport="http")` et consommé depuis d'autres langages ou clients.

## ⚙️ 4. Lire le serveur MCP local

Avant de connecter un agent, ouvrez `mcp_chinook.py` dans votre éditeur et repérez ses trois parties :

1. **La base** : connexion en lecture seule à `data/Chinook.db` via `sqlite3` ;
2. **`list_tables`** : renvoie la liste triée des tables ;
3. **`query_chinook`** : exécute un `SELECT` et renvoie jusqu'à 10 lignes.

La structure du serveur ressemble à ceci :

```python
mcp = FastMCP("chinook")

@mcp.tool()
def list_tables() -> list[str]:
    """Return the sorted list of tables available in the Chinook database."""
    ...

@mcp.tool()
def query_chinook(sql: str) -> str:
    """Execute a read-only SELECT (or WITH) query on Chinook..."""
    ...

if __name__ == "__main__":
    mcp.run()   # stdio par défaut · mcp.run(transport="http") pour HTTP
```

La cellule suivante vérifie que le fichier est présent, corrige un problème `stderr` spécifique à Windows/Jupyter, puis importe le client MCP.

### 🔮 Pause prédiction

En lisant `mcp_chinook.py`, quel argument `query_chinook` attend-elle ? Quel type retourne-t-elle ?

Formulez votre réponse avant de lancer la découverte MCP à la section 5.

In [4]:
import io
import sys
from pathlib import Path

# Vérification que le fichier serveur est bien présent dans ce dossier.
server_path = Path("mcp_chinook.py")
if not server_path.exists():
    raise FileNotFoundError(
        f"❌ Serveur MCP introuvable : {server_path.resolve()}. "
        "Lancez Jupyter depuis le dossier 'J2_Hands On' et redémarrez le kernel."
    )
print(f"✅ Serveur trouvé : {server_path.name}")


def stderr_has_fileno() -> bool:
    """Return whether the current stderr can be passed to a subprocess."""
    try:
        sys.stderr.fileno()
    except (AttributeError, io.UnsupportedOperation, OSError, ValueError):
        return False
    return True


if sys.platform == "win32" and not stderr_has_fileno():
    sys.stderr = sys.__stderr__
    print("⚙️ Compatibilité Windows activée pour le sous-processus MCP.")
else:
    print("✅ Aucun contournement stderr nécessaire.")

# 📚 L'import vient après le contrôle Windows afin que le SDK capture le bon stderr.
# Documentation officielle : https://docs.langchain.com/oss/python/langchain/mcp
from langchain_mcp_adapters.client import MultiServerMCPClient

✅ Serveur trouvé : mcp_chinook.py
⚙️ Compatibilité Windows activée pour le sous-processus MCP.


## 🛠️ 5. Décrire la connexion au serveur

Nous donnons à l'adaptateur une fiche de connexion :

- `command` : l'interpréteur Python courant (`sys.executable`) — garantit le bon environnement virtuel ;
- `args` : le fichier serveur à lancer ;
- `transport` : la manière d'échanger les messages.

À ce stade, nous ne choisissons encore aucun outil : nous indiquons seulement **où se trouve le serveur** et **comment lui parler**.

In [5]:
# 📚 MultiServerMCPClient : https://docs.langchain.com/oss/python/langchain/mcp
mcp_client = MultiServerMCPClient(
    {
        "chinook": {
            "transport": "stdio",
            "command": sys.executable,
            "args": ["mcp_chinook.py"],
        }
    }
)

print("✅ Configuration du serveur MCP prête.")

✅ Configuration du serveur MCP prête.


## 🔍 6. Découvrir les outils au lieu de les deviner

### 🔮 Pause prédiction

Combien d'outils le serveur expose-t-il ? Quels arguments attendent-ils ?

Vous avez lu `mcp_chinook.py` — vérifiez maintenant que ce que MCP annonce correspond exactement au code.

In [6]:
mcp_tools = await mcp_client.get_tools()

print(f"Nombre d'outils découverts : {len(mcp_tools)}")
for tool in mcp_tools:
    print(f"  • {tool.name} — {tool.description[:80]}")

Nombre d'outils découverts : 2
  • list_tables — Return the sorted list of tables available in the Chinook database.
  • query_chinook — Execute a read-only SELECT (or WITH) query on Chinook and return up to 10 rows.



### 👀 Ce qu'il faut observer

Vous devriez découvrir :

- `list_tables` — aucun argument, retourne la liste des tables ;
- `query_chinook` — attend `sql: str`, retourne une chaîne avec jusqu'à 10 lignes.

🔍 Le schéma joue le même rôle que pour notre outil local. La grande différence est son **origine** : il a été annoncé par le serveur pendant la connexion MCP — pas écrit dans ce notebook.

⚠️ `MultiServerMCPClient` est sans état par défaut : chaque appel d'outil ouvre une nouvelle session MCP, exécute l'action, puis la ferme.

## ▶️ 7. Appeler un outil MCP directement

Avant d'ajouter Mistral, isolons une seule pièce. Nous appelons directement `list_tables` pour lister les tables de Chinook.

### 🔮 Pause prédiction

Le résultat proviendra-t-il de LangChain, de Mistral ou du serveur MCP local ?

In [7]:
list_tables_tool = next(
    tool for tool in mcp_tools if tool.name == "list_tables"
)

tables = await list_tables_tool.ainvoke({})
print("Tables disponibles :", tables)

Tables disponibles : [{'type': 'text', 'text': 'Album', 'id': 'lc_af600594-26b6-4d2f-a77a-133ef2740aaa'}, {'type': 'text', 'text': 'Artist', 'id': 'lc_dae0651d-7fd9-4d6c-b578-9294acb15227'}, {'type': 'text', 'text': 'Customer', 'id': 'lc_340c84e8-6e8c-4235-9667-37ffe4336fcf'}, {'type': 'text', 'text': 'Employee', 'id': 'lc_d4c02d37-f851-4d00-b039-c1b4ae785449'}, {'type': 'text', 'text': 'Genre', 'id': 'lc_aa0c97f9-1527-4068-afcc-230a7872c7b8'}, {'type': 'text', 'text': 'Invoice', 'id': 'lc_669595c5-edcc-44c9-a1ec-4fa154df9aff'}, {'type': 'text', 'text': 'InvoiceLine', 'id': 'lc_ec5c77f5-8d0f-45e7-825b-dec8fda0836a'}, {'type': 'text', 'text': 'MediaType', 'id': 'lc_401c0b3b-c351-4cb1-a428-f9586637560d'}, {'type': 'text', 'text': 'Playlist', 'id': 'lc_8e4bcfe9-e18e-483b-8617-8bdec7dac073'}, {'type': 'text', 'text': 'PlaylistTrack', 'id': 'lc_08689ea7-967b-4a62-b8c5-d0be4c261e21'}, {'type': 'text', 'text': 'Track', 'id': 'lc_6eb3213e-efb7-4543-bef2-d47e71e0b8db'}]


### 👀 Lecture de la sortie

Le **serveur MCP local** (`mcp_chinook.py`) a interrogé SQLite. L'adaptateur a transporté la requête et converti le résultat. Mistral n'a toujours pas été appelé.

Cette étape est une technique de diagnostic très utile : si l'appel direct fonctionne mais pas l'agent, le serveur et le transport ne sont probablement pas la cause.

## 🤖 8. Confier le choix de l'outil à l'agent

[`create_agent`](https://docs.langchain.com/oss/python/langchain/agents) orchestre maintenant la boucle complète :

1. LangChain transmet à Mistral la question et les fiches des outils.
2. Mistral choisit un outil et produit ses arguments.
3. LangChain demande au serveur MCP d'exécuter l'outil.
4. `mcp_chinook.py` interroge Chinook et retourne le résultat.
5. LangChain renvoie ce résultat à Mistral pour formuler la réponse finale.

📚 Ce cycle correspond aux [cinq étapes du function calling documentées par Mistral](https://docs.mistral.ai/studio/conversations/function-calling). Le modèle **propose** l'appel ; il n'exécute pas lui-même la requête SQL.

In [8]:
from langchain.agents import create_agent

# 📚 create_agent : https://docs.langchain.com/oss/python/langchain/agents
agent_with_mcp = create_agent(
    model=mistral_model,
    tools=mcp_tools,
    system_prompt=(
        "Tu es un assistant qui répond aux questions sur la base musicale Chinook. "
        "Utilise list_tables si tu as besoin de connaître les tables disponibles, "
        "puis query_chinook pour interroger. N'exécute que des SELECT. Réponds en français."
    ),
)

### 🔮 Pause prédiction

Pour « Quel genre musical contient le plus de pistes ? », notez :

- le ou les outils que Mistral devrait choisir ;
- les arguments SQL probables ;
- l'ordre attendu des messages : humain → assistant avec appel → outil → assistant final.

In [9]:
result = await agent_with_mcp.ainvoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Quel genre musical contient le plus de pistes ?",
            }
        ]
    }
)

for index, message in enumerate(result["messages"], start=1):
    print(f"\n--- Message {index} · {type(message).__name__} ---")
    tool_calls = getattr(message, "tool_calls", None)
    if tool_calls:
        print("🧠 Appel(s) décidé(s) par Mistral :")
        print(json.dumps(tool_calls, indent=2, ensure_ascii=False))
    message.pretty_print()


--- Message 1 · HumanMessage ---
================================ Human Message =================================

Quel genre musical contient le plus de pistes ?

--- Message 2 · AIMessage ---
🧠 Appel(s) décidé(s) par Mistral :
[
  {
    "name": "list_tables",
    "args": {},
    "id": "Qy3FKTyw1",
    "type": "tool_call"
  }
]
================================== Ai Message ==================================
Tool Calls:
  list_tables (Qy3FKTyw1)
 Call ID: Qy3FKTyw1
  Args:

--- Message 3 · ToolMessage ---
================================= Tool Message =================================
Name: list_tables

[{'type': 'text', 'text': 'Album', 'id': 'lc_1b54a009-9b30-420b-9aaa-9053dc7cc19d'}, {'type': 'text', 'text': 'Artist', 'id': 'lc_ffe3723e-b983-4e14-951d-bd5b9dd6a20c'}, {'type': 'text', 'text': 'Customer', 'id': 'lc_753a5f03-5881-4d35-8358-d062654862f6'}, {'type': 'text', 'text': 'Employee', 'id': 'lc_4ba89328-cb44-4959-a826-452c261a61aa'}, {'type': 'text', 'text': 'Genre', 'id': 'lc_

### 🔍 Anatomie de ce qui vient de se passer

| Acteur | Responsabilité observée |
|---|---|
| 🧠 Mistral | choisit `query_chinook`, génère le SQL et rédige la réponse |
| 🔗 LangChain | maintient les messages et orchestre l'aller-retour |
| 🔌 Adaptateur MCP | convertit l'outil MCP en outil LangChain et transporte l'appel |
| 🧰 `mcp_chinook.py` | exécute réellement la requête SQL sur Chinook.db |
| 🐍 Python | fait tourner le client, l'agent et le sous-processus |

⚠️ Si aucun appel d'outil n'apparaît, ne confondez pas cela avec une panne MCP : le modèle peut avoir choisi de répondre seul. La consigne système réduit ce risque, mais le choix appartient toujours au modèle.

## 🧪 Micro-exercice — Quels artistes ont le plus d'albums ?

Complétez la cellule suivante pour appeler **directement** `query_chinook_tool` et trouver les 3 artistes avec le plus d'albums.

### ✅ Critères de réussite

- vous utilisez `await` et `.ainvoke(...)` ;
- l'argument s'appelle `sql` ;
- la requête joint `Artist` et `Album`, groupe par artiste et trie décroissant avec `LIMIT 3` ;
- la sortie liste au moins un artiste avec son nombre d'albums.

In [10]:
query_chinook_tool = next(
    tool for tool in mcp_tools if tool.name == "query_chinook"
)

# TODO 🧪 Décommentez et complétez la requête SQL.
# top_artists = await query_chinook_tool.ainvoke({"sql": "..."})
# print(top_artists)

<details>
<summary>✅ Afficher la correction</summary>

```python
query_chinook_tool = next(
    tool for tool in mcp_tools if tool.name == "query_chinook"
)
top_artists = await query_chinook_tool.ainvoke({
    "sql": (
        "SELECT ar.Name, COUNT(al.AlbumId) AS NbAlbums "
        "FROM Artist ar JOIN Album al ON ar.ArtistId = al.ArtistId "
        "GROUP BY ar.ArtistId ORDER BY NbAlbums DESC LIMIT 3"
    )
})
print(top_artists)
```

🔍 Ici, aucun modèle n'intervient : Python appelle l'outil LangChain, l'adaptateur dialogue en `stdio`, puis `mcp_chinook.py` interroge SQLite.
</details>

## ⚠️ Pièges fréquents et diagnostic

- **`mcp_chinook.py` introuvable** : lancez Jupyter depuis le dossier `J2_Hands On` et redémarrez le kernel.
- **`Connection closed`** : exécutez d'abord la cellule de découverte ; elle sépare un problème serveur/transport d'un problème de modèle.
- **Mauvais nom de table** : utilisez `list_tables` pour vérifier les noms exacts avant de requêter.
- **Confusion sur l'exécution** : Mistral génère une intention et des arguments SQL ; `mcp_chinook.py` exécute réellement la requête.
- **État MCP supposé** : le client est sans état par défaut. Une session persistante doit être demandée explicitement.
- **Windows/Jupyter** : le contournement `stderr` n'est appliqué que lorsque `fileno()` manque ; ne le copiez pas aveuglément dans une application classique.

## ✅ Ce que vous savez maintenant

Vous savez désormais que :

1. un outil local et un outil MCP présentent tous deux un contrat structuré à LangChain ;
2. un serveur MCP local se lit dans un fichier Python ordinaire (ici `mcp_chinook.py` avec `fastmcp`) ;
3. `MultiServerMCPClient.get_tools()` permet de découvrir les outils et leurs schémas ;
4. Mistral choisit l'outil, mais Python et le serveur l'exécutent réellement ;
5. un appel direct de l'outil aide à diagnostiquer chaque couche séparément.

## 🧭 Transition vers L6

Notre agent sait maintenant agir grâce à des outils externes. Mais si nous lui posons une deuxième question, se souviendra-t-il de la première ? Dans **L6 — Memory**, nous ajouterons une mémoire courte durée et vérifierons l'isolation entre conversations.

## 📚 Documentation officielle

- [LangChain — Model Context Protocol (MCP)](https://docs.langchain.com/oss/python/langchain/mcp)
- [LangChain — Tools](https://docs.langchain.com/oss/python/langchain/tools)
- [LangChain — Agents et `create_agent`](https://docs.langchain.com/oss/python/langchain/agents)
- [LangChain — Intégration `ChatMistralAI`](https://docs.langchain.com/oss/python/integrations/chat/mistralai)
- [Mistral AI — Les cinq étapes du function calling](https://docs.mistral.ai/studio/conversations/function-calling)
- [FastMCP — Documentation officielle](https://gofastmcp.com)